In [ ]:
%pip install pandas torch transformers datasets evaluate sacrebleu bert-score sentence-transformers rapidfuzz deep-translator tqdm numpy matplotlib PyYAML

In [ ]:
!git clone https://github.com/AbonentVneSeti/text_processing_final_project
%cd text_processing_final_project

In [1]:
# Ячейка 1: Импорты и загрузка конфига
import sys
sys.path.append('..')
from utils import load_config, build_dataloaders
import pandas as pd
import importlib

config = load_config("config.yaml")
print("Config loaded:")
print(config)

Config loaded:
{'dataset_source': 'tapaco', 'source_params': {'tapaco': {'language': 'ru', 'pair_mode': 'all_pairs'}, 'backtranslation': {'input_file': 'data/sentences.txt', 'from_lang': 'ru', 'to_lang': 'en', 'api': 'googletrans'}, 'custom_merge': {'files': ['data/ds1.csv', 'data/ds2.csv']}}, 'preprocessing': {'steps': ['remove_duplicates', 'normalize_text', 'filter_by_length', 'filter_edit_distance'], 'remove_duplicates': {}, 'normalize_text': {'lower': True, 'trim': True, 'remove_brackets': True}, 'filter_by_length': {'max_tokens': 128, 'tokenizer': 'cointegrated/rut5-base'}, 'filter_edit_distance': {'min_edit_ratio': 0.1, 'max_edit_ratio': 0.9}, 'filter_semantic_similarity': {'threshold': 0.75, 'model': 'sentence-transformers/LaBSE', 'batch_size': 32}}, 'model_name': 'rut5_paraphraser', 'model_config': {'pretrained_name': 'cointegrated/rut5-base', 'max_length': 128, 'batch_size': 16, 'learning_rate': '3e-4', 'num_epochs': 5, 'warmup_steps': 200, 'weight_decay': 0.01, 'gradient_accu

In [2]:
# Ячейка 2: Создание датасета
source = config["dataset_source"]
source_params = config["source_params"].get(source, {})
module = importlib.import_module(f"dataset.create_dataset.{source}")
df = getattr(module, "load_or_create")(source_params)
df = df.sample(n=40000, random_state=42)
print(f"Dataset size: {len(df)}")
df.head()

Dataset size: 40000


,original,paraphrase
6507,Он стал смеяться.,Я стал плакать.
263326,Погода облачная.,Начинается ночь.
646969,У меня нет абсолютно никаких проблем.,У меня вообще нет никаких проблем.
124858,Он худой.,Он поджарый.
126185,Ты чего вернулся?,Ты зачем вернулся?


In [3]:
# Ячейка 3: Предобработка
from dataset.prepare_dataset.prepare import prepare_dataset
df_clean = prepare_dataset(df, config["preprocessing"])
print(f"Clean dataset size: {len(df_clean)}")

Running filter_edit_distance: 100%|██████████| 4/4 [00:04<00:00,  1.09s/step]

Clean dataset size: 37468


In [4]:
# Ячейка 4: Построение даталоадеров
train_loader, val_loader, test_loader = build_dataloaders(df_clean, config["model_config"])
print(f"Train samples: {len(train_loader.dataset)}, Val: {len(val_loader.dataset)}, Test: {len(test_loader.dataset)}")

Train samples: 31847, Val: 4871, Test: 750


In [ ]:
# Ячейка 5: Создание модели и обучение
from models.rut5_paraphraser.model import ParaphraserModel
from models.trainer import train_model

model = ParaphraserModel(config["model_config"], device="cuda" if __import__('torch').cuda.is_available() else "cpu")
trained_model = train_model(
    model,
    train_loader,
    val_loader,
    config["model_config"],
    config["trainer_config"],
    config["metrics_config"]
)
trained_model.save(config["trainer_config"]["output_dir"])

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!cp -r "models/rut5_paraphraser/saves" "/content/drive/MyDrive/colab_export/"

from google.colab import runtime
runtime.unassign()

In [5]:
 
# Ячейка 6: Оценка на тесте
from models.model_use import load_model, evaluate_model

model = load_model(
    config["model_name"],
    config["model_config"],
    checkpoint_path=config["trainer_config"]["output_dir"]
)

test_metrics = evaluate_model(model, test_loader, config["metrics_config"])
print("Test metrics:", test_metrics)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Test metrics: {'bleu': 16.869804239981704, 'bertscore': 0.786685321058266, 'cosine_similarity': 0.84145784}
